In [1]:
!pip install torch-scatter torch-sparse torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.8 MB/s eta 0:00:0000:01


In [28]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from tqdm import tqdm
import random
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

EMBED_DIM = 64
N_LAYERS = 4
BATCH_SIZE = 2048
EPOCHS = 15
LR = 1e-3

PATIENCE = 3
MIN_DELTA = 1e-4

In [29]:
DATA_PATH = "/kaggle/input/datasets/organizations/grouplens/movielens-20m-dataset"

ratings = pd.read_csv(f"{DATA_PATH}/rating.csv")

ratings = ratings[ratings['rating'] >= 4.0]
ratings = ratings[['userId','movieId','timestamp']]
ratings = ratings.sort_values(['userId','timestamp'])

In [30]:
user_map = {u:i for i,u in enumerate(ratings['userId'].unique())}
item_map = {i:j for j,i in enumerate(ratings['movieId'].unique())}

ratings['user'] = ratings['userId'].map(user_map)
ratings['item'] = ratings['movieId'].map(item_map)

num_users = len(user_map)
num_items = len(item_map)

In [31]:
train_data, val_data, test_data = [], [], []

for user, group in ratings.groupby('user'):
    items = group['item'].values
    if len(items) < 3:
        train_data.extend([(user, i) for i in items])
    else:
        train_data.extend([(user, i) for i in items[:-2]])
        val_data.append((user, items[-2]))
        test_data.append((user, items[-1]))

In [32]:
from scipy.sparse import coo_matrix, diags, vstack, hstack

rows = [u for u,i in train_data]
cols = [i for u,i in train_data]

R = coo_matrix((np.ones(len(rows)), (rows, cols)),
               shape=(num_users, num_items))

upper = hstack([coo_matrix((num_users, num_users)), R])
lower = hstack([R.T, coo_matrix((num_items, num_items))])

adj_mat = vstack([upper, lower]).tocoo()

rowsum = np.array(adj_mat.sum(axis=1)).flatten()
d_inv = np.power(rowsum, -0.5)
d_inv[np.isinf(d_inv)] = 0.

D_inv = diags(d_inv)
norm_adj = (D_inv @ adj_mat @ D_inv).tocoo()

indices = torch.LongTensor(np.vstack([norm_adj.row, norm_adj.col]))
values = torch.FloatTensor(norm_adj.data)

norm_adj = torch.sparse.FloatTensor(indices, values, torch.Size(norm_adj.shape)).coalesce().to(device)

/tmp/ipykernel_55/1598388730.py:15: RuntimeWarning: divide by zero encountered in power
  d_inv = np.power(rowsum, -0.5)


In [33]:
user_pos = defaultdict(set)
for u,i in train_data:
    user_pos[u].add(i)

def sample_batch(batch_size):
    users = np.random.randint(0, num_users, batch_size)
    pos, neg = [], []
    
    for u in users:
        p = random.choice(list(user_pos[u]))
        n = np.random.randint(0, num_items)
        while n in user_pos[u]:
            n = np.random.randint(0, num_items)
        pos.append(p)
        neg.append(n)

    return (
        torch.LongTensor(users).to(device),
        torch.LongTensor(pos).to(device),
        torch.LongTensor(neg).to(device)
    )

In [36]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, dim, layers):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.layers = layers

        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)

        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, adj):
        emb = torch.cat([self.user_emb.weight, self.item_emb.weight])
        embs = [emb]

        for _ in range(self.layers):
            emb = torch.sparse.mm(adj, emb)
            embs.append(emb)

        embs = torch.stack(embs, dim=1)
        final = torch.mean(embs, dim=1)

        return torch.split(final, [self.n_users, self.n_items])

In [37]:
def bpr_loss(user_emb, item_emb, users, pos, neg):
    u = user_emb[users]
    p = item_emb[pos]
    n = item_emb[neg]

    pos_scores = (u*p).sum(dim=1)
    neg_scores = (u*n).sum(dim=1)

    loss = -torch.mean(torch.log(torch.sigmoid(pos_scores - neg_scores)))

    reg = (u.norm(2)**2 + p.norm(2)**2 + n.norm(2)**2) / users.shape[0]

    return loss + 1e-4 * reg

In [38]:
def evaluate(data, k=20):
    model.eval()
    user_emb, item_emb = model(norm_adj)

    recall, ndcg = [], []

    for u, gt in data:
        scores = torch.matmul(user_emb[u], item_emb.T)
        scores[list(user_pos[u])] = -1e9

        topk = torch.topk(scores, k).indices

        if gt in topk:
            recall.append(1)
            rank = (topk == gt).nonzero(as_tuple=True)[0].item()
            ndcg.append(1 / np.log2(rank + 2))
        else:
            recall.append(0)
            ndcg.append(0)

    return np.mean(recall), np.mean(ndcg)

In [ ]:
model = LightGCN(num_users, num_items, EMBED_DIM, N_LAYERS).to(device)
optimizer = Adam(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_ndcg = 0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for _ in tqdm(range(len(train_data)//BATCH_SIZE)):
        users, pos, neg = sample_batch(BATCH_SIZE)

        # recompute every batch (correct autograd)
        user_emb, item_emb = model(norm_adj)

        loss = bpr_loss(user_emb, item_emb, users, pos, neg)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    val_recall, val_ndcg = evaluate(val_data)

    print(f"Epoch {epoch+1} Loss {total_loss:.4f} | NDCG {val_ndcg:.4f}")

    if val_ndcg > best_ndcg + MIN_DELTA:
        best_ndcg = val_ndcg
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
        print("Best model saved")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{PATIENCE})")

        if patience_counter >= PATIENCE:
            print(" Early stopping")
            break

100%|██████████| 4746/4746 [1:21:42<00:00,  1.03s/it]


Epoch 1 Loss 628.9871 | NDCG 0.0370
Best model saved


100%|██████████| 4746/4746 [1:21:42<00:00,  1.03s/it]


Epoch 2 Loss 474.5187 | NDCG 0.0414
Best model saved


100%|██████████| 4746/4746 [1:21:44<00:00,  1.03s/it]


...

100%|██████████| 4746/4746 [1:21:40<00:00,  1.03s/it]


Epoch 5 Loss 307.6668 | NDCG 0.0457
Best model saved


100%|██████████| 4746/4746 [1:21:36<00:00,  1.03s/it]


Epoch 6 Loss 288.2842 | NDCG 0.0465
Best model saved


 63%|██████▎   | 2997/4746 [51:29<30:02,  1.03s/it]  

In [1]:
model.load_state_dict(torch.load("best_model.pth"))

test_recall, test_ndcg = evaluate(test_data)

print("FINAL TEST")
print("Recall@20:", test_recall)
print("NDCG@20:", test_ndcg)

NameError: name 'model' is not defined

In [ ]:
import shutil, pickle

SAVE_DIR = "/kaggle/working/gnn_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

model.eval()
user_emb, item_emb = model(norm_adj)

torch.save(model.state_dict(), f"{SAVE_DIR}/model.pth")

np.save(f"{SAVE_DIR}/user_emb.npy", user_emb.cpu().detach().numpy())
np.save(f"{SAVE_DIR}/item_emb.npy", item_emb.cpu().detach().numpy())

with open(f"{SAVE_DIR}/user_map.pkl","wb") as f:
    pickle.dump(user_map,f)

with open(f"{SAVE_DIR}/item_map.pkl","wb") as f:
    pickle.dump(item_map,f)

shutil.make_archive("/kaggle/working/gnn_outputs", 'zip', SAVE_DIR)

print("Download from Output panel")

In [ ]:
import os
import shutil
import pickle

SAVE_DIR = "/kaggle/working/gnn_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

# Ensure embeddings exist
model.eval()
user_emb, item_emb = model(norm_adj)

# Save artifacts
torch.save(model.state_dict(), f"{SAVE_DIR}/model.pth")
np.save(f"{SAVE_DIR}/user_emb.npy", user_emb.cpu().detach().numpy())
np.save(f"{SAVE_DIR}/item_emb.npy", item_emb.cpu().detach().numpy())

with open(f"{SAVE_DIR}/user_map.pkl","wb") as f:
    pickle.dump(user_map,f)

with open(f"{SAVE_DIR}/item_map.pkl","wb") as f:
    pickle.dump(item_map,f)

# Zip everything
zip_path = "/kaggle/working/gnn_outputs.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', SAVE_DIR)

print(" Ready for download:")
print(zip_path)